# Code Generation Agent

- Use a GPU runtime: **Runtime → Change runtime type → T4 GPU**.
- Put the model and the dataset in your Google Drive (see Setup), then **Runtime → Run all**.

## 1. Setup

The agent runs **Phi-3.5-mini-instruct** from Hugging Face: https://huggingface.co/microsoft/Phi-3.5-mini-instruct

1. Download the model into a folder named `phi_3_5_mini_instruct`.

2. Download the dataset CSV from [Kaggle](https://www.kaggle.com/datasets/aliiihussain/amazon-sales-dataset?resource=download) (free account).

3. Upload both anywhere in your Google Drive, then set their paths as `MODEL_PATH` and `DATA_PATH` at the top of the agent cell. Drive paths start with `/content/drive/MyDrive/`; after the first cell mounts Drive, you can right-click a file in the **Files** panel → **Copy path**.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip -q install bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.4 MB/s eta 0:00:00


## 2. Agent

The `src/agent` package in one cell, in this order: `config`, `data`, `llm`, `state`, `parsing`, `prompts`, `policy`, `classify`, `mapping`, `codegen`, `guards`, `present`, `loop`. `__main__` is in the Example cell below. The `from agent... import` lines are left out, since everything shares one namespace here, and `MODEL_PATH` / `DATA_PATH` point to Google Drive.

In [4]:
MODEL_PATH = "/content/drive/MyDrive/hf-models/phi_3_5_mini_instruct"
LOAD_IN_4BIT = True  # Ignored when no GPU is available

DATA_PATH = "/content/drive/MyDrive/hf-models/agent_project/amazon_sales_dataset.csv"

ALLOWED_ACTIONS = [
    "classify_request",
    "run_analysis",
    "reject_request",
    "answer_user",
    "finish",
]

MAX_RESULT_ROWS = 50
MAX_RESULT_COLS = 3

MAX_STEPS = 10
MAX_DECISION_ATTEMPTS = 3
MAX_ATTEMPTS = 2


import pandas as pd


def load_data(path=DATA_PATH):
    """Load the dataset and normalise the date column.

    to_datetime keeps nanosecond precision, which makes every row unique and
    gets order_date classified as an identifier, and excluded from results that could cause data leakage.
    so we used floor('D') which drops the time part
    """
    df = pd.read_csv(path)
    df["order_date"] = pd.to_datetime(df["order_date"]).dt.floor("D")
    return df


_model = None
_tokenizer = None


def _load_local():
    """Loads on the first call only, the model stays in memory afterwards."""
    global _model, _tokenizer

    if _model is not None:
        return

    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

    _tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)

    kwargs = {
        "device_map": "auto",
        "local_files_only": True,
    }

    if LOAD_IN_4BIT and torch.cuda.is_available():
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
        )

    _model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, **kwargs)
    device = "GPU" if torch.cuda.is_available() else "CPU"
    print(f"Model loaded from {MODEL_PATH} ({device})")


def ask_llm(messages, max_new_tokens=256):
    _load_local()

    import torch

    prompt = _tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = _tokenizer(prompt, return_tensors="pt")
    device = _model.get_input_embeddings().weight.device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = _model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=_tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return _tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


import uuid


def new_state():
    """Create a fresh state for one request."""
    return {
        "request_id": str(uuid.uuid4()),
        "request_received": False,
        "request_classified": False,
        "authorized": None,          # None means "not judged yet".
        "analysis_done": False,
        "result": None,
        "answered": False,
        "rejection_reason": None,
        "finished": False,
        "policy_override": None,
        "history": [],

        # Request decomposition
        "request_parts": [],
        "denied_parts": [],
        "question_to_run": None,
        "classifier_error": False,

        # Column mapping
        "map_stage_failure": None,

        # Raw model output and repair attempts
        "column_map": None,
        "map_attempts": [],
        "code_attempts": [],
    }


def state_for_llm(state):
    """Return only the state fields needed for the next action."""
    return {
        "request_classified": state["request_classified"],
        "authorized": state["authorized"],
        "analysis_done": state["analysis_done"],
        "answered": state["answered"],
    }


import ast
import json
import re
import textwrap


class ExtractionFailure(Exception):
    """The model returned no valid Python code."""
    pass


def extract_json(text):
    """Extract the first valid JSON object from the text."""

    decoder = json.JSONDecoder()

    for i, ch in enumerate(text):
        if ch == "{":
            try:
                obj, _ = decoder.raw_decode(text[i:])
                return obj
            except json.JSONDecodeError:
                continue

    raise ValueError("No valid JSON object found.")


def extract_code(text):
    text = textwrap.dedent(text).strip()

    # 1. Prefer fenced code.
    blocks = re.findall(
        r"```(?:python)?\s*(.*?)```",
        text,
        re.DOTALL | re.IGNORECASE
    )

    if blocks:
        code = max(blocks, key=len).strip()

        # Validate fenced code before accepting it.
        try:
            ast.parse(code)
        except SyntaxError as e:
            raise ExtractionFailure(
                f"Fenced code is not valid Python: {e}"
            ) from e

        return code

    # 2. Accept the whole output if it is valid Python.
    try:
        ast.parse(text)
        return text
    except SyntaxError:
        pass

    # 3. Otherwise, try to keep the longest valid prefix.
    lines = [line for line in text.splitlines() if line.strip()]

    for end in range(len(lines), 0, -1):
        candidate = "\n".join(lines[:end])

        try:
            ast.parse(candidate)
            return candidate
        except SyntaxError:
            continue

    raise ExtractionFailure("Could not extract valid Python code.")


def build_prompt(system_prompt, user_prompt, context=None):
    """Shape only: system + user.
    Context (schema, state, columns) is appended to the system message
    """

    if context:
        system_prompt = f"{system_prompt}\n\n{context}"

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt}
    ]


DECISION_PROMPT = """
You are the controller of a data analytics agent.

Your job is to choose the NEXT single action. You do NOT write code
and you do NOT answer the question yourself.

Available actions:
- classify_request  : inspect the request before doing anything
- run_analysis      : generate and execute pandas code
- reject_request    : refuse the request
- answer_user       : turn the computed result into a sentence
- finish            : end the workflow

STATE INTERPRETATION:

- If request_classified is False (authorized will be None):
  the next action MUST be classify_request.

- If request_classified is True and authorized is False:
  the next action MUST be reject_request.

- If request_classified is True and authorized is True
  and analysis_done is False:
  the next action MUST be run_analysis.

- If analysis_done is True and answered is False:
  the next action MUST be answer_user.

- If answered is True:
  the next action MUST be finish.

Rules:
- Output ONLY a JSON object.
- No explanations, no markdown, no code fences.
- The JSON must contain exactly these keys:
  {"action": "<one of the actions above>"}
- Choose exactly ONE action.
- Never choose run_analysis before classify_request has run.
- Never choose reject_request before classify_request has run.
- Never choose answer_user before run_analysis has run.
- Choose finish only after the user has received an answer or a rejection.

"""


def build_decision_prompt(question, state, error=None):

    context = f"Current state:\n{state_for_llm(state)}"

    if error:
        context += (
            f"\n\nYour previous output was rejected: {error}\n"
            f"Try again and follow the rules exactly."
        )

    return build_prompt(DECISION_PROMPT, user_prompt=question, context=context)


CLASSIFY_PROMPT = """
You are a request classifier for a data analytics agent.

Split the request into independent sub-requests, then decide for EACH part
whether it is allowed.

SPLITTING RULES:
- Each part's text MUST be copied verbatim from the request. Never rewrite,
  summarize, translate, or rephrase.
- Split ONLY where there are genuinely independent sub-requests.
- If the request contains one single task, return exactly one part.
- Do NOT separate a qualifier, constraint, modifier, or condition from the
  request it applies to. Examples of such qualifiers:
    raw / before aggregation / per record / show each /
    as it appears in the dataset

- Do NOT split on "and", "by", or "per" when the text after them names
  a column to group by. A grouping phrase describes HOW to aggregate
  the same request; it is not an independent sub-request.
- If the request contains one single task, return exactly one part.
- Do NOT separate a qualifier, constraint, modifier, or condition from the
  request it applies to.


- If a qualifier asks to show, list, or export individual records, return
  that qualifier as its OWN part marked not allowed. The remaining request
  may be allowed on its own.

Allowed:
- totals
- averages
- counts
- distributions
- group-by summaries
- trends
- aggregated statistics

Rejected:
- raw records
- individual records
- representative examples drawn from records
- concrete occurrences from the dataset
- samples of actual transactions
- table export or full dataset view
- any modification of the dataset (update, delete)
- any request whose answer requires exposing a specific dataset record

Output ONLY a JSON object:
{"parts": [{"text": "<verbatim excerpt>", "allowed": true or false, "reason": "<short reason>"}]}
"""


# No context, and that is the decision: the classifier judges the WORDING
def build_classify_prompt(question):
    return build_prompt(CLASSIFY_PROMPT, user_prompt=question)


# The retry corrects FORMAT only
CLASSIFY_RETRY = """
FORMAT CORRECTION ONLY.

Your previous classification was NOT rejected. Your judgement about
what is allowed was accepted and must not change.

What failed was the FORMAT of one field: {error}

The "text" of every part must be COPIED CHARACTER BY CHARACTER from
the request. Do not correct spelling. Do not fix grammar. Do not tidy
the wording. If the request contains a typo, copy the typo.

Return the SAME parts with the SAME "allowed" values. Change only the
"text" fields so they match the request exactly.
"""


def build_classify_retry_prompt(question, error):
    return build_prompt(
        CLASSIFY_PROMPT + "\n\n" + CLASSIFY_RETRY.format(error=error),
        user_prompt=question,
    )


MAP_PROMPT = """
Map each concept in the question onto a column of the DataFrame.

You do NOT write code and you do NOT answer the question.

FIRST, SPLIT THE QUESTION INTO CONCEPTS:

- Each key represents ONE data concept that the user is asking about.

- A key is NOT the name of an operation.
  Operations describe what should be done with the data later.
  Do not include the requested operation in a concept key.

- A key is NOT a DataFrame column name.
  The key represents the concept from the user's request.
  The actual DataFrame column is written in "column".

- A key is NOT a phrase copied from the question, and never describes
  the whole calculation. Identify the underlying data concepts first,
  then write one key per concept.

- When a concept is expressed using several words, use the underlying
  concept as the key rather than copying the entire phrase into the key.

THEN, FOR EACH CONCEPT:

- Scan the ENTIRE column list before deciding. Do not stop at the first
  name that looks right.
- Identify every existing column that could hold the data for that
  concept.
- A concept may be DERIVED from a source column. A "day of week" concept
  is answered by whatever column holds the date, not by a column named
  day_of_week. Do not require the concept itself to exist as a column.
- Copy column names EXACTLY from the list below.
- Never invent a name. Every name you write, in "column" or in
  "candidates", MUST appear in the list below.
- A column existing in the DataFrame does NOT prove it is the one meant.

- Exactly one existing column is a plausible source:
    "column": that column,  "candidates": [],  "certain": true

- More than one existing column is a plausible source:
    "column": "",  "candidates": [all of them],  "certain": false

- NEVER put the selected "column" inside "candidates".
- "candidates" holds only the alternatives you did NOT pick.

Output ONE JSON object containing ALL concepts. Not one object per concept.
No explanations, no markdown, no code fences.

{"rating": {"column": "rating", "candidates": [], "certain": true},
 "category": {"column": "product_category", "candidates": [], "certain": true}}

{"orders": {"column": "order_id", "candidates": [], "certain": true},
 "payment_method": {"column": "payment_method", "candidates": [], "certain": true}}

{"price": {"column": "", "candidates": ["price", "discounted_price"], "certain": false}}

{"profit": {"column": "", "candidates": [], "certain": false}}

"""

def build_map_prompt(question, columns):
    context = "Available columns:\n" + "\n".join(columns)
    return build_prompt(MAP_PROMPT, user_prompt=question, context=context)



# SINGLE-CONCEPT MAPPING

CONCEPT_PROMPT = """
Which column of the DataFrame holds the data for the concept below?

- Scan the ENTIRE column list before deciding.
- A concept may be DERIVED from a column rather than stored in one. Any
  part of a date — day, week, month, quarter, year — is answered by the
  column that holds the date. Answer with the source column and set
  "certain": true; do not require the concept itself to exist as a column.

- Copy the column name EXACTLY from the list. Never invent a name.

Answer in two lines, in this order.

First line — set aside the operation being requested (average, count,
total, highest) and look at the data the question is about. Does a column
hold that data, or can it be computed from one? Related subject matter is
not enough: the column has to be the data itself or produce it.
    exists: yes
    exists: no

Second line — the JSON object:
- Exactly one plausible column:
    {"column": "<name>", "candidates": [], "certain": true}
- More than one plausible column:
    {"column": "", "candidates": [<all of them>], "certain": false}
- No column fits:
    {"column": "", "candidates": [], "certain": false}

Output only those two lines. No explanations, no code fences.
"""

def build_concept_prompt(concept, columns):
    context = "Available columns:\n" + "\n".join(columns)
    return build_prompt(CONCEPT_PROMPT, user_prompt=f"Concept: {concept}", context=context)


CODE_PROMPT = """
You are a data analyst working with a pandas DataFrame named df.

STRICT RULES:

1. DATA ACCESS
- Use ONLY existing column names exactly as written.
- Use the provided DataFrame `df` as-is.
- Do NOT redefine, replace, or recreate `df`.
- Do NOT invent columns, rows, values, or external data.
- The column map below tells you which column each concept in the
  question refers to. Use those columns.
- A column whose dtype is object is NOT a datetime column. Before using
  .dt on it, convert it with pd.to_datetime(...).

- A requested concept may be derived from a mapped source column.
  If the concept does not exist as a column but can be derived from the
  mapped column, derive it instead of looking for or inventing another column.

- When deriving a value from a mapped column, use the mapped source column
  directly. Do not require the derived concept to exist as a DataFrame column.

- A date-part concept means the part, not the calendar date:
  day of week → .dt.day_name()   month → .dt.month
  quarter     → .dt.quarter      year  → .dt.year
  Never use pd.Grouper or resample for a date-part grouping.


2. CODE RESTRICTIONS
- Do NOT write any import statement.
- `pd` is already available in the execution environment.
- Use `pd` directly when pandas functionality is needed.
- Never write `import pandas as pd`.
- Do NOT define functions.
- Do NOT use loops.
- Use pandas only.
- Do NOT call print. Assign the answer to `result` only.
- Do NOT call reset_index(). Assign the grouped Series to `result` as it is.

3. OUTPUT PRIVACY
- NEVER return raw rows or a full table view.
- NEVER expose individual records or person-level data.
- Return only aggregated results: totals, averages, counts, or group summaries.

4. OUTPUT CONTRACT
- Store the final answer in a variable named `result`.
- Output ONLY executable Python code.

These rules apply regardless of how the request is phrased.
"""


def build_code_prompt(question, schema, column_map=None):
    """The analysis path always passes column_map, CODE_PROMPT refers to
    it, and the rule means nothing without it."""

    context = f"Dataset schema:\n{schema}"

    if column_map:
        lines = [
            f"- {k}  →  {v['column']}  ({v['dtype']})"
            for k, v in column_map.items()
        ]
        context += "\n\nColumn map:\n" + "\n".join(lines)

    return build_prompt(
        CODE_PROMPT,
        user_prompt=question,
        context=context
    )


# REPAIR

CODE_REPAIR_PROMPT = """
You are a pandas debugger.

Fix the code based on the error.

Rules:
- Return ONLY corrected python code.
- No explanations.

The corrected code must still obey ALL of the following:

1. DATA ACCESS
- Use ONLY existing column names exactly as written.
- Use the provided DataFrame `df` as-is.
- Do NOT redefine, replace, or recreate `df`.
- Do NOT invent columns, rows, values, or external data.

2. CODE RESTRICTIONS
- Do NOT add an import statement.
- `pd` is already available.
- Preserve the original operation and meaning.
- Do not replace a grouping operation with a different time granularity just to avoid an error.
- Do NOT define functions.
- Do NOT use loops.
- Use pandas only.
- Do NOT call print. Assign the answer to `result` only.

3. OUTPUT PRIVACY
- NEVER return raw rows or a full table view.
- NEVER expose individual records or person-level data.
- Return only aggregated results: totals, averages, counts, or group summaries.

4. OUTPUT CONTRACT
- Store the final answer in a variable named `result`.
- Output ONLY executable Python code.
"""


def build_code_repair_prompt(schema, question, code, error):
    """Argument order follows the order published by the Lab: repair_prompt(schema, question, code, error)"""

    context = f"Dataset schema:\n{schema}"

    return build_prompt(
        CODE_REPAIR_PROMPT,
        user_prompt=(
            f"Question:\n{question}\n\n"
            f"Broken Code:\n{code}\n\n"
            f"Error:\n{error}\n\n"
            f"Return fixed code:"
        ),
        context=context
    )


def _apply_policy(action, state):

    # Classification must happen before rejecting or analyzing.
    if action == "reject_request" and not state["request_classified"]:
        state["policy_override"] = (
            "reject_request → classify_request (not classified yet)"
        )
        return "classify_request"

    if action == "run_analysis" and not state["request_classified"]:
        state["policy_override"] = (
            "run_analysis → classify_request (not classified yet)"
        )
        return "classify_request"

    # Never analyze a request that was classified as unauthorized.
    if action == "run_analysis" and state["authorized"] is False:
        state["policy_override"] = (
            "run_analysis → reject_request (unauthorized)"
        )
        return "reject_request"

    # An answer requires a completed analysis result.
    if action == "answer_user" and state["result"] is None:
        state["policy_override"] = (
            "answer_user → run_analysis (no result yet)"
        )
        return "run_analysis"

    # The workflow cannot finish before answering or rejecting.
    if action == "finish" and not (state["answered"] or state["rejection_reason"]):
        state["policy_override"] = (
            "finish → classify_request (nothing done yet)"
        )
        return "classify_request"

    # Do not repeat a successful analysis.
    if action == "run_analysis" and state["result"] is not None:
        state["policy_override"] = (
            "run_analysis → answer_user (analysis already done)"
        )
        return "answer_user"

    # Do not answer more than once.
    if action == "answer_user" and state["answered"]:
        state["policy_override"] = (
            "answer_user → finish (already answered)"
        )
        return "finish"

    # Do not classify a request that is already classified.
    if action == "classify_request" and state["request_classified"]:
        if state["authorized"] is False:
            state["policy_override"] = (
                "classify_request → reject_request (already classified, unauthorized)"
            )
            return "reject_request"

        state["policy_override"] = (
            "classify_request → run_analysis (already classified)"
        )
        return "run_analysis"

    return action


def enforce_policy(action, state, max_passes=6):
    """Apply policy rules until the action is stable or the pass limit is reached."""

    for _ in range(max_passes):

        new_action = _apply_policy(action, state)

        if new_action == action:
            return action

        action = new_action

    # Conflicting rules must not leave the workflow in an unknown state.
    state["policy_override"] = "unstable policy resolution → reject_request"
    return "reject_request"


# Phrases that always make a part unsafe.
DENY_PHRASES = [
    "all rows", "all records", "every transaction",
    "entire dataset", "full table", "raw data",
    "example order", "example record", "example transaction",
    "example row", "example customer",
    "sample row", "sample record", "sample order",
    "actual transaction", "actual order", "actual record",
]


def _ask_classifier(question):
    """Call the classifier and return its parts or an error."""

    try:
        raw = ask_llm(build_classify_prompt(question))
        print("\n       CLASSIFIER RAW OUTPUT:")
        print("      " + raw.replace("\n", "\n      "))
        return extract_json(raw)["parts"], None
    except Exception as e:
        print(f"       classifier error: {e}")
        return None, f"{type(e).__name__}: {e}"



def validate_parts(question, parts):
    """Deterministic validation of the classifier output contract."""

    if not isinstance(parts, list) or not parts:
        return False, "NO_PARTS: no parts returned"

    for p in parts:
        if not isinstance(p, dict):
            return False, "NOT_OBJECT: part is not an object"

        if not isinstance(p.get("text"), str) or not p["text"].strip():
            return False, "NO_TEXT: part has no text"

        if not isinstance(p.get("allowed"), bool):
            return False, "ALLOWED_TYPE: 'allowed' is not a boolean"

    return True, None


def _apply_deny_phrases(parts):
    """Override the model when a known unsafe phrase is detected."""
    for p in parts:
        hit = next((ph for ph in DENY_PHRASES if ph in p["text"].lower()), None)
        if hit and p["allowed"]:
            p["allowed"] = False
            p["reason"] = f"deny phrase: '{hit}'"
            print(f"        deny-phrase override on part: {p['text'][:50]!r}")


def _fail_closed(state, question, err):
    """Reject the request when the classifier output cannot be trusted."""

    state["classifier_error"] = True
    state["authorized"] = False
    state["request_parts"] = []
    state["denied_parts"] = [{"text": question, "allowed": False, "reason": err}]
    state["question_to_run"] = None
    state["rejection_reason"] = (
        "Could not analyse the request. Please rephrase it more clearly."
    )
    print(f"       classifier output failed : {err}")
    return state



def _derive_state(state, question, parts):
    """Build the final state and the text allowed to reach code generation."""
    state["request_parts"] = parts
    state["denied_parts"] = [p for p in parts if not p["allowed"]]
    state["authorized"] = any(p["allowed"] for p in parts)

    allowed_texts = [p["text"] for p in parts if p["allowed"]]

    if not allowed_texts:
        # Nothing safe remains, so no analysis is allowed.
        state["question_to_run"] = None
        state["rejection_reason"] = "; ".join(p["reason"] for p in parts)

    elif not state["denied_parts"]:
        # If nothing was removed, keep the original request unchanged.
        state["question_to_run"] = question
        state["rejection_reason"] = None

    else:
        # Run only the allowed parts.
        state["question_to_run"] = "\n".join(
            f"{i + 1}. {t}" for i, t in enumerate(allowed_texts)
        )
        state["rejection_reason"] = None


def _report(state, parts):
    """Print a short classification summary for debugging."""
    print(f"       split into {len(parts)} part(s) → "
          f"authorized = {state['authorized']}")
    for p in parts:
        mark = "[allow]" if p["allowed"] else "[deny] "
        print(f"         {mark} {p['text'][:60]}  — {p['reason']}")


def classify_request(question, state):
    """Split the request, judge each part, and derive the state. Two
    deterministic guards sit around the model: validate_parts checks the
    contract, DENY_PHRASES overrides specific phrases. Any contract failure
    fails closed."""

    state["request_classified"] = True

    parts, err_detail = _ask_classifier(question)

    if parts is None:
        ok, err = False, err_detail or "classifier output could not be parsed"

    # A missing or non-text reason would crash the report
    for p in parts:
        if not isinstance(p.get("reason"), str):
            p["reason"] = "no reason given"

    else:
        ok, err = validate_parts(question, parts)

    if not ok:
        return _fail_closed(state, question, err)

    _apply_deny_phrases(parts)
    _derive_state(state, question, parts)
    _report(state, parts)

    return state


import re
import json


# 1. EXCEPTIONS
# ______________________________________________________________________________________

class MapStageFailure(Exception):
    """Base for every exception that happens BEFORE code generation."""
    pass


class MappingFailure(MapStageFailure):
    """The model's map could not be parsed or broke the contract. Not repaired."""
    pass


class MultipleCandidates(MapStageFailure):
    """A concept matches more than one column."""
    pass


class LowMappingConfidence(MapStageFailure):
    """The map is fine, but the model wasn't sure: it either picked a column
    and said it's not certain (certain=false), or found no column for the
    concept at all."""
    pass



# 2. HELPERS
# ______________________________________________________________________________________


def _print_map_raw(raw, indent="      "):
    """Pretty-print the map output for reading only."""
    try:
        parsed = json.loads(raw)
    except Exception:
        print(indent + raw.replace("\n", "\n" + indent))
        return

    pretty = json.dumps(parsed, indent=2, ensure_ascii=False)
    print(indent + pretty.replace("\n", "\n" + indent))


def _pre_split_question(question):
    """Split the question into data and grouping parts."""
    for sep_word in ["by", "per"]:
        pattern = rf'(.+?)\s+{sep_word}\s+(.+)'
        match = re.match(pattern, question, re.IGNORECASE)
        if match:
            data = match.group(1).strip()
            dims = [d.strip() for d in re.split(r'\s+and\s+', match.group(2))]
            return data, dims
    return None, []


def _exact_column_in_key(key, columns):
    """Find a column name that appears exactly in the key.

    If multiple columns match, return the longest one.
    Return None if there is no clear match.
    """
    k = key.replace("_", " ").lower()
    hits = [c for c in columns
            if re.search(rf"\b{re.escape(c.replace('_', ' ').lower())}\b", k)]
    if not hits:
        return None
    longest = max(hits, key=len)
    l = longest.replace("_", " ").lower()
    if all(h.replace("_", " ").lower() in l for h in hits):
        return longest
    return None



# 3. BUILD: split the question into concepts, bind each to a column
# ______________________________________________________________________________________

def _map_by_concept(concepts, columns, state):
    """PATH 1: Map each concept to one column."""
    column_map = {}

    for concept in concepts:
        raw = ask_llm(build_concept_prompt(concept, columns))
        state["map_attempts"].append(raw)
        print(f"\n      🗺️  CONCEPT {concept!r} RAW OUTPUT:")
        _print_map_raw(raw)

        # The model states its judgement BEFORE the JSON, where the JSON is
        # still bound by it. Measured: writing the JSON first, it asserts a
        # column and then concedes the gap in trailing prose, which
        # extract_json never reads.
        if re.search(r"\bexists:\s*no\b", raw, re.IGNORECASE):
            raise LowMappingConfidence(
                f"No column holds or yields {concept!r}. "
                "Please name the column explicitly."
            )

        try:
            column_map[concept] = extract_json(raw)
        except Exception as e:
            raise MappingFailure(
                f"Could not map {concept!r}: {type(e).__name__}: {e}"
            )

    return column_map


def _map_whole_question(question, columns, state):
    """PATH 2: Map the whole question to its columns."""
    raw = ask_llm(build_map_prompt(question, columns))
    state["map_attempts"].append(raw)
    print("\n      🗺️  COLUMN MAP RAW OUTPUT:")
    _print_map_raw(raw)

    try:
        return extract_json(raw)
    except Exception as e:
        raise MappingFailure(
            f"Could not map this question: {type(e).__name__}: {e}"
        )


def _split_and_bind(question, df, state):
    """Choose the mapping path based on the question."""
    columns = list(df.columns)
    data_concept, group_dims = _pre_split_question(question)

    if group_dims:
        return _map_by_concept([data_concept] + group_dims, columns, state)
    return _map_whole_question(question, columns, state)


# 4. CORRECT — deterministic fixes to the model's raw map
# ______________________________________________________________________________________

def _normalise_map(column_map):
    """Fix the shape in code instead of asking the model again.
    Clean extra candidates when a column is already chosen.
    """

    if not isinstance(column_map, dict):
        return

    for entry in column_map.values():
        if not isinstance(entry, dict):
            continue
        col = entry.get("column")
        cands = entry.get("candidates")

        if not isinstance(cands, list):
            continue

        if col and cands == [col]:
            entry["candidates"] = []

        if col and cands and cands != [col]:
            entry["candidates"] = []


def _apply_exact_name_rule(column_map, columns):
    """A key that literally contains a column name is a certain match: the
    user wrote the name, so the model's hesitation (price vs discounted_price)
    isn't a real choice. Set the column, clear candidates, mark certain."""
    for key, entry in column_map.items():
        col = _exact_column_in_key(key, columns)
        if col:
            entry["column"], entry["candidates"], entry["certain"] = col, [], True



# 5. VALIDATE — structure, then existence, then decision
# ______________________________________________________________________________________

def _check_map_structure(column_map):
    """Check the shape: every entry has the three fields with the right types,
    and column/candidates are not both filled."""

    if not isinstance(column_map, dict) or not column_map:
        return "invalid", "mapping is empty or not an object"

    for key, entry in column_map.items():
        if not isinstance(entry, dict):
            return "invalid", f"{key!r} is not an object"
        if not isinstance(entry.get("column"), str):
            return "invalid", f"{key!r}: 'column' is not a string"
        if not isinstance(entry.get("candidates"), list):
            return "invalid", f"{key!r}: 'candidates' is not a list"
        if not all(isinstance(c, str) for c in entry["candidates"]):
            return "invalid", f"{key!r}: 'candidates' holds a non-string"
        if not isinstance(entry.get("certain"), bool):
            return "invalid", f"{key!r}: 'certain' is not a boolean"

        # candidates + certain=false means the model declared ambiguity.
        # That is allowed here, the DECISION stage handles it.
        if entry["candidates"] and entry["certain"] is False:
            continue

        if entry["column"] and entry["candidates"]:
            return "invalid", (
                f"{key!r}: 'column' and 'candidates' cannot both be set. "
                f"If {entry['column']!r} is the answer, set candidates to []. "
                f"If it is not certain, set column to \"\"."
            )

    return None


def _check_map_existence(column_map, df):
    """Every name the model wrote must be a real column."""
    for key, entry in column_map.items():
        col = entry["column"]
        if col and col not in df.columns:
            return "invalid", f"{key!r}: column does not exist: {col!r}"

        # Candidates that don't exist + certain=false means the model found
        # nothing, not that it broke the contract.
        if entry["certain"] is False and entry["candidates"] \
                and not any(c in df.columns for c in entry["candidates"]):
            return "uncertain", f"{key!r} has no matching column"

        for c in entry["candidates"]:
            if c not in df.columns:
                return "invalid", f"{key!r}: candidate does not exist: {c!r}"

    return None


def _check_map_decision(column_map):
    """The map is usable only if every concept has one clear column. Ambiguity
    is checked first: it also leaves the column empty, and would otherwise be
    reported as a missing column."""
    for key, entry in column_map.items():
        if len(entry["candidates"]) > 1:
            names = ", ".join(entry["candidates"])
            return "ambiguous", f"Not sure which column {key!r} means: {names}"

    for key, entry in column_map.items():
        if not entry["column"]:
            return "uncertain", f"{key!r} has no usable column"
        if entry["certain"] is False:
            return "uncertain", f"Not confident that {key!r} means {entry['column']!r}"

    return None


def validate_column_map(column_map, df):
    """Structure, then existence, then decision. The order is a CORRECTNESS
    CONDITION, not organisation: a declared ambiguity has an empty "column",
    so a combined check would read it as a missing column and return the
    wrong reason. Each stage returns."""

    verdict = _check_map_structure(column_map)
    if verdict:
        return verdict

    verdict = _check_map_existence(column_map, df)
    if verdict:
        return verdict

    verdict = _check_map_decision(column_map)
    if verdict:
        return verdict

    return "valid", None


def _raise_for_verdict(kind, reason):
    """One exception type per stop, so the agent loop can count them apart."""
    if kind == "ambiguous":
        raise MultipleCandidates(f"{reason}. Please name the column explicitly.")
    if kind == "uncertain":
        raise LowMappingConfidence(f"{reason}. Please name the column explicitly.")
    if kind == "invalid":
        raise MappingFailure(
            "Could not map this question onto the dataset columns. "
            "Please rephrase it using the column names."
        )



# 6. ENRICH — attach real dtypes after validation
# ______________________________________________________________________________________

def _attach_dtypes(column_map, df):
    """dtype comes from the DataFrame, never from the model. Added after
    validation so the validator stays read-only."""
    for entry in column_map.values():
        entry["dtype"] = str(df[entry["column"]].dtype)



# 7. ORCHESTRATOR — the four steps in order
# ______________________________________________________________________________________

def resolve_columns(question, df, state):
    """Four steps: split & bind → correct → validate → enrich.

    Mapping is never repaired: low confidence or multiple matches stop the
    request."""

    columns = list(df.columns)

    # 1. split & bind (two paths)
    column_map = _split_and_bind(question, df, state)

    # 2. correct (deterministic, no model call)
    _normalise_map(column_map)
    _apply_exact_name_rule(column_map, columns)

    # 3. validate (stops on failure)
    kind, reason = validate_column_map(column_map, df)
    _raise_for_verdict(kind, reason)

    # 4. enrich (dtypes, after validation)
    _attach_dtypes(column_map, df)

    print(f"        mapped {len(column_map)} concept(s)")
    for key, entry in column_map.items():
        print(f"         {key} → {entry['column']}  ({entry['dtype']})")

    # Recorded AFTER dtype is added: this is the map the code prompt
    # actually receives, not the one the validator judged.
    state["column_map"] = column_map
    return column_map


import re

def _clean_error(e, limit=300):
    """Error text for the repair prompt. Long quoted strings are data values
    pandas copied into the message: replace them so data does not reach the
    model, and keep the rest so the model still sees what went wrong."""
    msg = re.sub(r"'[^']{40,}'", "'<data omitted>'", str(e))
    return f"{type(e).__name__}: {msg}"[:limit]

def ask_llm_to_fix(schema, question, code, error):
    return ask_llm(build_code_repair_prompt(schema, question, code, error))


def _attempt(code, column_map, df, attempt, max_attempts):
    """Run one execution attempt and return the result or error."""

    try:
        code = extract_code(code)
    except ExtractionFailure as e:
        # Save the raw output before extraction so failed code is still recorded.
        if attempt == max_attempts - 1:
            raise
        print(f"       Not valid Python: {e}")
        return None, f"SyntaxError: the code is not valid Python. {e}"

    print(f"\n       Attempt {attempt + 1}/{max_attempts}")
    print("      Generated code:")
    for line in code.splitlines():
        print("        ", line)

    try:
        result = run_generated_code(code, df)
        print("       Execution success")
        return result, None

    except PolicyViolation:
        raise

    except Exception as e:
        print(f"       Failed: {e}")

        # Keep the actual error so the final failure explains what went wrong.
        if attempt == max_attempts - 1:
            raise RuntimeError(f"Max attempts reached — last error: {e}")

        return None, _clean_error(e)


def run_code_agent_with_retry(question, df, state, max_attempts=MAX_ATTEMPTS):
    """Generate, run, and repair code within a fixed attempt limit."""

    # Include small text values as stored.
    # Only category labels with up to 10 distinct values are included.
    small_text = {
        c: sorted(df[c].dropna().unique().tolist())
        for c in df.columns
        if df[c].dtype == object and df[c].nunique() <= 10
    }

    schema = {
        "columns": list(df.columns),
        "rows": len(df),
        "values": small_text,
    }

    column_map = resolve_columns(question, df, state)
    code = ask_llm(build_code_prompt(question, schema, column_map))

    for attempt in range(max_attempts):

        # Record the original model output before extraction.
        state["code_attempts"].append(code)

        result, error = _attempt(code, column_map, df, attempt, max_attempts)
        if error is None:
            return result

        print("       Asking the model to repair the code...")
        code = ask_llm_to_fix(schema, question, code, error)

    raise RuntimeError("Retry loop ended without a result")


import ast
import pandas as pd


# CONSTANTS
# _______________________________________________________________________

FORBIDDEN_NODES = (
    ast.Import, ast.ImportFrom,
    ast.With,
    ast.While, ast.For,
    ast.ListComp, ast.DictComp, ast.SetComp, ast.GeneratorExp,
    ast.Try,
    ast.FunctionDef, ast.ClassDef,
    ast.Delete,
)

FORBIDDEN_NAMES = {
    "exec", "eval", "open", "__import__", "compile",
    "os", "sys", "subprocess", "shutil", "print", "getattr",
}

# pandas can turn the whole DataFrame into a string with to_csv(),
# which can bypass the shape check so we block this at the source.
_FORBIDDEN_CALL_ATTRS = {
    "read_csv", "read_excel", "read_json", "read_parquet", "read_pickle",
    "read_sql", "read_html", "read_table", "read_clipboard", "read_feather",
    "read_stata", "read_sas", "read_spss",
    "to_csv", "to_excel", "to_json", "to_parquet", "to_pickle", "to_sql",
    "to_clipboard", "to_feather", "to_hdf", "to_string", "to_markdown",
    "to_html", "to_latex", "to_xml",
}


ALLOWED_BUILTINS = {
    "len": len, "min": min, "max": max, "sum": sum,
    "sorted": sorted, "round": round,
    "int": int, "float": float,   # added after cases failed where the model used
                                  # int()/float() in code and the sandbox rejected them
}

_ROW_COUNT_OPERATION_WORDS = {"count", "number", "num", "how many"}
_AGG_METHODS = {"count", "nunique", "sum", "mean", "size", "min", "max"}
_LIST_CONVERSIONS = {"tolist", "to_list", "to_dict"}


# EXCEPTIONS
# _______________________________________________________________________

# PolicyViolation stops the request (not repairable).
# ValueError goes to the repair loop.

class PolicyViolation(Exception):
    """The result breaks the output policy.
    It is stopped before or after execution and is never repaired."""
    pass


# HELPER METHODS
# _______________________________________________________________________

# Small shared helper functions.
# - identifier_columns & key_signals_row_count: shared with mapping.py.
# - _column_was_aggregated: internal use only.


def identifier_columns(df):
    """Columns where every value is unique."""
    return {c for c in df.columns if df[c].is_unique}


def key_signals_row_count(concept_key: str) -> bool:
    """True if the concept key asks for a row count. distinct/unique need a real column,
    so they return False even though they mention counting."""

    key_lower = concept_key.lower().replace("_", " ")
    if "distinct" in key_lower or "unique" in key_lower:
        return False
    return any(word in key_lower for word in _ROW_COUNT_OPERATION_WORDS)



def _column_was_aggregated(tree, column):
    """True if an aggregation was applied to this column.

    A count can inherit the column's name, df.count()['order_id'] is
    labeled 'order_id' but holds counts, not ids. Without this check, the
    identifier guard would flag that count as a leak and reject it.

    Three forms checked:
        df["col"].count()   — subscript, then aggregate
        df.col.count()      — attribute, then aggregate
        df.count()["col"]   — aggregate, then subscript
    """
    # form 1: aggregation applied to the column
    for node in ast.walk(tree):
        if not (isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute)):
            continue
        if node.func.attr not in _AGG_METHODS:
            continue

        target = node.func.value
        if isinstance(target, ast.Subscript):
            sl = target.slice
            if isinstance(sl, ast.Constant) and sl.value == column:
                return True


        if isinstance(target, ast.Attribute) and target.attr == column:
            return True

    # form 2: the column selected out of an aggregated frame
    for node in ast.walk(tree):
        if not isinstance(node, ast.Subscript):
            continue
        sl = node.slice
        if not (isinstance(sl, ast.Constant) and sl.value == column):
            continue
        inner = node.value
        if (isinstance(inner, ast.Call) and isinstance(inner.func, ast.Attribute)
                and inner.func.attr in _AGG_METHODS):
            return True

    return False


# CHECKS THAT READ THE CODE (before execution)
# _______________________________________________________________________


def validate_code_safety(code: str):
    """Check the code before running it."""

    tree = ast.parse(code)

    for node in ast.walk(tree):

        if (isinstance(node, ast.Name)
                and node.id == "df"
                and isinstance(node.ctx, ast.Store)):
            raise ValueError(
                "Forbidden: `df` cannot be reassigned. The DataFrame is "
                "already loaded — read from it, do not rebuild it."
            )

        if isinstance(node, FORBIDDEN_NODES):
            raise ValueError(f"Forbidden operation: {type(node).__name__}")

        if isinstance(node, ast.Name) and node.id in FORBIDDEN_NAMES:
            raise ValueError(f"Forbidden name used: {node.id}")

        if isinstance(node, ast.Attribute) and node.attr.startswith("__"):
            raise ValueError(f"Forbidden attribute: {node.attr}")

        if (isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute)
                and node.func.attr in _FORBIDDEN_CALL_ATTRS):
            raise ValueError(f"Forbidden call: {node.func.attr}")


def validate_result_conversion(code):
    """Reject code that converts the result to a list (.tolist() or list())
    lists has no name or index for the shape check to read, so the model
    just needs to drop the conversion.

    Caught here as a ValueError (repairable) rather than later in
    validate_result_shape as a PolicyViolation (non-repairable)
    """
    tree = ast.parse(code)
    for node in ast.walk(tree):
        if (isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute)
                and node.func.attr in _LIST_CONVERSIONS):
            raise ValueError(
                f"Do not convert the result with .{node.func.attr}(): assign the "
                "Series, DataFrame, or scalar itself to `result`."
            )
        if (isinstance(node, ast.Call) and isinstance(node.func, ast.Name)
                and node.func.id == "list"):
            raise ValueError(
                "Do not wrap the result in list(): assign the Series, "
                "DataFrame, or scalar itself to `result`."
            )


def validate_self_reference(code):
    """Reject code that subtracts a column from itself (col - col), which
    can happen when the model reaches for a missing column and silently
    produces a fake result

    Two forms:
        df["col"] - df["col"]        — the `-` operator
        df["col"].sub(df["col"])     — the .sub()/.subtract() method
    """
    tree = ast.parse(code)
    cols = lambda n: {s.value for s in ast.walk(n)
                      if isinstance(s, ast.Constant) and isinstance(s.value, str)}
    for node in ast.walk(tree):
        if isinstance(node, ast.BinOp) and isinstance(node.op, ast.Sub):
            if cols(node.left) & cols(node.right):
                raise ValueError(
                    "The code subtracts a column from itself; this dataset "
                    "does not contain what the question asks for."
                )
        if (isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute)
                and node.func.attr in {"sub", "subtract"}):
            left_cols = cols(node.func.value)
            arg_cols = set().union(
                *(cols(a) for a in node.args),
                *(cols(kw.value) for kw in node.keywords)
            ) if (node.args or node.keywords) else set()
            if left_cols & arg_cols:
                raise ValueError(
                    "The code subtracts a column from itself; this dataset "
                    "does not contain what the question asks for."
                )


def validate_group_keys(code, df):
    """Reject a groupby whose key has more than MAX_RESULT_ROWS distinct
    values, grouping by a near-unique column can create one row per record.
    The key is checked before the code runs, even if the result is later
    limited with .head(50).

    Only direct column names are checked here. Derived keys like (df['order_date'].dt.month)
    are checked later by validate_result_shape.
    """
    tree = ast.parse(code)

    for node in ast.walk(tree):
        if not (isinstance(node, ast.Call)
                and isinstance(node.func, ast.Attribute)
                and node.func.attr == "groupby"):
            continue

        keys = []
        for arg in node.args:
            if isinstance(arg, ast.Constant) and isinstance(arg.value, str):
                keys.append(arg.value)
            elif isinstance(arg, (ast.List, ast.Tuple)):
                keys += [e.value for e in arg.elts
                         if isinstance(e, ast.Constant) and isinstance(e.value, str)]

        keys = [k for k in keys if k in df.columns]
        if not keys:
            continue                     # derived key — out of scope

        n = 1
        for k in keys:
            n *= df[k].nunique()

        if n > MAX_RESULT_ROWS:
            raise PolicyViolation(
                f"Aggregated output only — grouping by {keys} yields {n} groups. "
                "Try a higher-level grouping — for example by region or by category."
            )


# CHECKS THAT READ THE RESULT (after execution)
# _______________________________________________________________________

# The result must not use identifier columns as values or group keys.
# Otherwise, the grouping can produce one row per record.
# validate_result_shape runs these checks in order:
#   _check_size    — checks the number of rows and columns
#   _check_columns — checks the result columns
#   _check_index   — checks the group key


def _check_size(result):
    """Reject a result with too many rows or columns.
    A result this large is likely to contain individual records.
    """
    rows = len(result)
    cols = result.shape[1] if isinstance(result, pd.DataFrame) else 1

    if rows > MAX_RESULT_ROWS:
        raise PolicyViolation(
            f"Aggregated output only — result has {rows} rows "
            f"(limit {MAX_RESULT_ROWS})."
        )
    if cols > MAX_RESULT_COLS:
        raise PolicyViolation(
            f"Aggregated output only — result has {cols} columns "
            f"(limit {MAX_RESULT_COLS}); this looks like raw records."
        )


def _is_identifier(name, values, df, ids, id_values, tree):
    """Does this column/index level hold record identifiers?

    An aggregated column is exempted even when its values fall in an id
    column's range, a count can equal an id by coincidence, so we trust
    _column_was_aggregated (the code), not the values.

    Known gap: see decisions.md.
    """
    if tree is not None and name is not None and _column_was_aggregated(tree, name):
        return False
    if name in ids:
        return True
    if name is None or name not in df.columns:
        return False

    vals = set(pd.Series(values).dropna().unique())
    if not vals:
        return False
    if pd.Series(values).dtype != df[name].dtype:
        return False

    result_dtype = pd.Series(values).dtype
    for id_col, id_vals in id_values.items():
        if df[id_col].dtype != result_dtype:
            continue
        if vals <= id_vals:
            return True
    return False


def _check_columns(result, df, ids, id_values, tree):
    """Check if any column holds identifiers (ids in the values).
    Size alone isn't enough: head(50) on a per-order breakdown is still 50
    individual orders, what matters is what each row is."""
    if isinstance(result, pd.DataFrame):
        for name in result.columns:
            if _is_identifier(name, result[name].values, df, ids, id_values, tree):
                raise PolicyViolation(
                    f"Aggregated output only — '{name}' identifies "
                    f"individual records."
                )
    elif _is_identifier(result.name, result.values, df, ids, id_values, tree):
        raise PolicyViolation(
            "Aggregated output only — the result identifies individual records."
        )


def _check_index(result, df, ids, id_values):
    """Check if the index holds identifiers (ids in the index, not the values).

    After a groupby the key lives in the index, so this catches a derived
    key like .dt.month that validate_group_keys couldn't check before
    execution. Such a key is allowed when its values are computed: .dt.month
    gives an index named 'order_date' holding month numbers, not dates, the
    dtype check below tells them apart (int months != datetime column) and
    lets it pass. (If the model calls reset_index() the key becomes a column
    instead, and _check_columns catches it there.)

    KNOWN GAP (same trade-off as _is_identifier, see decisions.md):
    """
    idx = result.index
    if isinstance(idx, pd.RangeIndex):
        return

    for lvl in range(idx.nlevels):
        name = idx.names[lvl]
        level = idx.get_level_values(lvl)

        if name is None or name not in df.columns:
            continue

        # consider int and float as one family
        src = df[name]
        same_family = (pd.Series(level).dtype == src.dtype) or (
            pd.api.types.is_numeric_dtype(level)
            and pd.api.types.is_numeric_dtype(src)
        )
        if not same_family:
            continue

        vals = set(pd.Series(level).dropna().unique())
        if name in ids or (vals and any(vals <= v for v in id_values.values())):
            raise PolicyViolation(
                "Aggregated output only — the index identifies "
                "individual records."
            )


def validate_result_shape(result, df, code=None):
    """Size, then identifiers by name, then the index. Stops on the
    result's shape raise PolicyViolation and are never repaired. None and
    method objects raise ValueError because they are code mistakes the
    repair loop can fix.
    """
    if isinstance(result, (pd.DataFrame, pd.Series)):
        _check_size(result)

        ids = identifier_columns(df)
        if not ids:
            return

        id_values = {c: set(df[c].dropna().unique()) for c in ids}

        tree = ast.parse(code) if code is not None else None
        _check_columns(result, df, ids, id_values, tree)
        _check_index(result, df, ids, id_values)
        return


    if result is None:
        raise ValueError(
            "`result` is None: assign the computed Series, DataFrame, "
            "or scalar to `result`. An in-place operation "
            "(inplace=True) returns None."
        )

    if callable(result):
        raise ValueError(
            "`result` is a method, not a value: call it with (), "
            "for example .nunique() instead of .nunique"
        )

    if pd.api.types.is_scalar(result):
        return


    if pd.api.types.is_scalar(result):
        return

    raise PolicyViolation(
        "Aggregated output only — the result must be a Series, a "
        f"DataFrame, or a plain scalar. Got {type(result).__name__}, "
        "which carries no column or index to check."
    )



# THE RUNNER
# _______________________________________________________________________



def run_generated_code(code, df):
    """Run the model's code in a restricted sandbox.
    Code checks run before execution, and result checks run after execution.
    """

    # before: everything that reads the code
    validate_code_safety(code)
    validate_result_conversion(code)
    validate_self_reference(code)
    validate_group_keys(code, df)

    # execution
    safe_globals = {
        "__builtins__": ALLOWED_BUILTINS,
        "df": df.copy(),
        "pd": pd,
    }
    safe_locals = {}
    exec(code, safe_globals, safe_locals)

    if "result" not in safe_locals:
        raise ValueError("Code must assign a `result` variable.")

    # after: the only check that reads the result
    result = safe_locals["result"]
    validate_result_shape(result, df, code=code)
    return result


import pandas as pd


def _fmt(v, decimals=4, sig=4):
    """Shorten floats for display without changing the stored result."""

    if isinstance(v, bool) or not isinstance(v, float):
        return v

    if v != v or v in (float("inf"), float("-inf")):
        return v

    if v == 0:
        return 0.0

    if abs(v) >= 1:
        r = round(v, decimals)
    else:
        from math import floor, log10
        r = round(v, sig - int(floor(log10(abs(v)))) - 1)

    return int(r) if r == int(r) else r


def format_result(question, result):
    """Format analysis results for display without changing the result."""

    if isinstance(result, pd.Series):
        head = f"{result.name or 'value'} by {result.index.name or 'group'}:"
        body = "\n".join(
            f"         {k}: {_fmt(v)}"
            for k, v in result.items()
        )
        return f"{head}\n{body}"

    if isinstance(result, pd.DataFrame):
        shown = result.copy()

        for c in shown.columns:
            if shown[c].dtype.kind == "f":
                shown[c] = shown[c].map(_fmt)

        body = shown.to_string(
            index=not isinstance(shown.index, pd.RangeIndex)
        ).replace("\n", "\n         ")

        return f"{len(result)} rows:\n         {body}"

    if isinstance(result, dict):
        # Dicts are typically produced by .to_dict().
        items = [f"{k}: {_fmt(v)}" for k, v in result.items()]
        return (
            items[0] + "".join(f"\n         {i}" for i in items[1:])
            if items
            else "{}"
        )

    return str(_fmt(result))


import pandas as pd
import time


def validate_action(action):
    """Check whether the model returned a valid action."""
    return action in ALLOWED_ACTIONS


def decide(question, state, max_attempts=MAX_DECISION_ATTEMPTS, verbose=False):
    """Ask the model for an action, retrying when the output is invalid."""

    # Reset attempts for this decision so old attempts are not mixed with new ones.
    state["_decision_attempts"] = []
    error = None

    for _ in range(max_attempts):

        messages = build_decision_prompt(question, state, error)
        raw = ask_llm(messages)
        state["_decision_attempts"].append(raw)

        if verbose:
            print(f"       LLM raw: {raw}")

        try:
            decision = extract_json(raw)
        except Exception:
            error = "output was not valid JSON."
            print("        decision retry — output was not valid JSON")
            continue

        action = decision.get("action")

        if not validate_action(action):
            error = f"'{action}' is not an allowed action."
            print(f"        decision retry — '{action}' is not an allowed action")
            continue

        return decision

    return None


def execute_action(action, state, df, question):

    if action == "classify_request":
        classify_request(question, state)

    elif action == "run_analysis":
        target = state["question_to_run"] or question
        state["result"] = run_code_agent_with_retry(target, df, state)
        state["analysis_done"] = True

    elif action == "reject_request":
        if not state["rejection_reason"]:
            state["rejection_reason"] = "Request not permitted."
        print("\n       Request Rejected – Unauthorized Query")
        print(f"         {state['rejection_reason']}")
        state["finished"] = True

    elif action == "answer_user":
        print(f"\n       Raw result: {state['result']}")
        print(f"       {format_result(state['question_to_run'] or question, state['result'])}")

        if state["denied_parts"]:
            print("\n       Not executed:")
            for p in state["denied_parts"]:
                print(f"         - {p['text']}  ({p['reason']})")

        state["answered"] = True

    elif action == "finish":
        state["finished"] = True

    return state




def _log_step(state, step, raw_action, action):
    """Record the model's action, final action, overrides, and decision attempts."""
    state["history"].append({
        "step": step + 1,
        "llm_action": raw_action,
        "final_action": action,
        "override": state["policy_override"],
        "decision_attempts": state.pop("_decision_attempts", []),
    })


# Exceptions that should stop the workflow and be displayed to the users.
_STOPS = (
    (PolicyViolation,   None),
    (MapStageFailure,   None),
    (ExtractionFailure, "Could not produce runnable code for this question. "
                        "Please rephrase it."),
    (RuntimeError,      "Could not compute safely: {error}"),
)


def _handle_stop(state, exc, verbose):
    """Record a stop and end the run. Returns False when the exception is
    not one the loop handles, so the caller re-raises it."""
    for exc_type, template in _STOPS:
        if not isinstance(exc, exc_type):
            continue

        state["rejection_reason"] = (
            str(exc) if template is None else template.format(error=exc)
        )
        if isinstance(exc, MapStageFailure):
            # Store which mapping stage failed.
            state["map_stage_failure"] = type(exc).__name__
        state["finished"] = True

        if verbose:
            print(f"\n       {type(exc).__name__}: {exc}")
        return True

    return False



def print_run_summary(state):
    overrides = sum(
        1 for h in state["history"]
        if h["override"]
    )

    print("\n   ── Run Summary ──")
    print(f"   request_id: {state['request_id']}")
    print(f"   steps: {len(state['history'])}")
    print(f"   policy overrides: {overrides}")
    print(f"   mapping attempts: {len(state['map_attempts'])}")
    print(f"   code attempts: {len(state['code_attempts'])}")
    print(f"   duration: {state['duration_seconds']:.2f}s")
    print(f"   answered: {state['answered']}")


def run_agent(question, df, verbose=True):

    state = new_state()
    start = time.perf_counter()
    state["request_received"] = True

    if verbose:
        print("\n" + "=" * 62)
        print(f" {question}")
        print("=" * 62)

    for step in range(MAX_STEPS):

        decision = decide(question, state)

        if decision is None:
            # Keep failed decision attempts in the history for diagnosis.
            _log_step(state, step, None, None)
            state["rejection_reason"] = "Invalid decision from model."
            state["finished"] = True
            if verbose:
                print("\n    Model failed to produce a valid decision.")
            break

        raw_action = decision["action"]
        state["policy_override"] = None
        action = enforce_policy(raw_action, state)

        if verbose:
            print(f"\n   ── Step {step + 1} " + "─" * 40)
            print(f"    LLM chose      : {raw_action}")
            if state["policy_override"]:
                print(f"     Policy override: {state['policy_override']}")
            print(f"    Executing      : {action}")

        _log_step(state, step, raw_action, action)

        # Keep stage failures in the state so the final run can be diagnosed.
        try:
            state = execute_action(action, state, df, question)

        except Exception as e:
            if not _handle_stop(state, e, verbose):
                raise
            break

        if state["finished"]:
            break

    # The loop may reach MAX_STEPS without finishing normally.
    if not state["finished"]:
        state["rejection_reason"] = (
            f"Workflow did not settle within {MAX_STEPS} steps."
        )
        state["finished"] = True

    state["duration_seconds"] = time.perf_counter() - start


    if verbose:
        overrides = sum(1 for h in state["history"] if h["override"])
        print("\n   " + "─" * 46)
        print(f"   steps: {len(state['history'])}   overrides: {overrides}   "
              f"answered: {state['answered']}")
        print("=" * 62)

    if verbose:
        print_run_summary(state)

    return state



def solve(question: str, df: pd.DataFrame, verbose=True):
    """Returns the result on success, or the rejection reason as a string."""

    state = run_agent(question, df, verbose=verbose)

    if state["rejection_reason"]:
        return state["rejection_reason"]

    if not state["answered"]:
        print("\n     Workflow ended without answering.")
        return "Could not complete this request."

    return state["result"]

## 3. Example

Asks for one question and prints the result, like `python -m agent`. Run the cell again to ask another question; the model stays loaded.

In [ ]:
def main():
    df = load_data()
    question = input("What would you like to know? ")
    r = solve(question, df)

    if isinstance(r, str):
        print(f"\n{r}")
    else:
        print("\nResult:")
        print(r)


if __name__ == "__main__":
    main()

What would you like to know? avg revenue by category

 avg revenue by category


[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


Loading weights:   0%|          | 0/195 [00:01<?, ?it/s]

Model loaded from /content/drive/MyDrive/hf-models/phi_3_5_mini_instruct (CPU)
